# Eye Disease Classification — ResNet18 (reconstructed, reproducible workflow)

**Status:** This notebook is a reconstruction of the workflow used in the previous Kaggle session. **The original trained `.pth` checkpoints and original grouped split CSVs were not recovered.** Running this notebook creates NEW splits and trains NEW weights; its metrics need not reproduce the historical results in `results/`.

Stages: data preparation and group-aware splitting → exploratory plots → frozen-backbone ResNet18 → layer4 fine-tuning → sqrt-reweighted fine-tuning → held-out test → optional TTA validation. **Run the held-out test only after final model/TTA decisions are frozen.**

**Before running:** Kaggle notebook Settings → Accelerator → GPU. Add your authorized original dataset ZIP or extracted dataset through Kaggle Add Input. Edit `DATA_ZIP`/`DATA_DIR` in cell 2. Do not commit image data or large model weights to GitHub.

In [ ]:
# 1. Imports and reproducibility
import os, re, json, copy, time, random, zipfile
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import functional as TF, InterpolationMode
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
ARTIFACTS = Path('/kaggle/working/eye_disease_outputs')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. Dataset location — CHANGE THESE PATHS if necessary
# For a mounted Kaggle dataset: DATA_DIR = Path('/kaggle/input/YOUR-DATASET/eye_dataset')
# For the ZIP previously downloaded into /kaggle/working:
DATA_ZIP = Path('/kaggle/working/eye_dataset.zip')
DATA_DIR = Path('/kaggle/working/eye_dataset')

if not DATA_DIR.exists() and DATA_ZIP.exists():
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall('/kaggle/working')

# Recover a nested eye_dataset/ if the archive has a wrapper directory.
if not (DATA_DIR / 'metadata.csv').exists():
    matches = list(Path('/kaggle/working').glob('**/metadata.csv'))
    if len(matches) == 1:
        DATA_DIR = matches[0].parent

assert (DATA_DIR / 'metadata.csv').exists(), (
    'metadata.csv missing: attach/extract the original dataset and update DATA_DIR.'
)
IMAGE_DIR = DATA_DIR / 'images'
assert IMAGE_DIR.exists(), 'images directory missing'
metadata = pd.read_csv(DATA_DIR / 'metadata.csv')
required = {'filename', 'dataset', 'unified_label'}
assert required.issubset(metadata.columns), f'Missing columns: {required - set(metadata.columns)}'
print('Dataset:', DATA_DIR, 'images:', len(metadata))
display(metadata.head())

## Group-aware 70/10/20 split

The historic split files `splits_v2/*.csv` were lost with the prior Kaggle session. If you already have those exact files, set `EXISTING_SPLITS_DIR` below and use them. Otherwise the code creates *new*, approximately stratified splits while keeping both eyes from each identifiable **ODIR** patient in the same split. For APTOS, ACRIMA, and ORIGA, grouping is filename-level because verified patient identifiers are not available in the recovered metadata. This does **not** establish independence of all patients or rule out cross-source duplicates.

In [ ]:
# 3. Load original historic splits if available; otherwise reconstruct grouped splits
EXISTING_SPLITS_DIR = None  # e.g. '/kaggle/input/my-splits/splits_v2'
SPLIT_DIR = ARTIFACTS / 'splits_v2'
SPLIT_DIR.mkdir(exist_ok=True)

if EXISTING_SPLITS_DIR is not None:
    folder = Path(EXISTING_SPLITS_DIR)
    train_df, val_df, test_df = [pd.read_csv(folder / f'{x}.csv') for x in ['train','val','test']]
    print('Using supplied original split CSVs.')
else:
    df = metadata.copy()
    # ODIR_<patient_id>_left/right; if the naming pattern differs, inspect before training.
    def group_id(row):
        name = str(row['filename'])
        source = str(row['dataset']).upper()
        if source.startswith('ODIR'):
            m = re.match(r'ODIR[_-](\d+)[_-](?:left|right)', name, flags=re.I)
            if m: return f'ODIR_patient_{m.group(1)}'
            raise ValueError(f'Unrecognized ODIR patient filename: {name}')
        return f'{source}__{name}'
    df['group_id'] = df.apply(group_id, axis=1)
    # A group may span multiple labels. Stratify using the group's most frequent label.
    grp = (df.groupby('group_id').agg(dataset=('dataset', 'first'),
                                     label=('unified_label', lambda s: s.value_counts().index[0]))
             .reset_index())
    grp['stratum'] = grp['dataset'].astype(str) + '||' + grp['label'].astype(str)
    # Fall back to source stratification if source/class strata are too small.
    if grp['stratum'].value_counts().min() < 2:
        grp['stratum'] = grp['dataset'].astype(str)
    try:
        build, holdout = train_test_split(grp, test_size=0.20, random_state=SEED, stratify=grp['stratum'])
    except ValueError:
        build, holdout = train_test_split(grp, test_size=0.20, random_state=SEED)
    try:
        inner_train, inner_val = train_test_split(build, test_size=0.125,
                                                  random_state=SEED,
                                                  stratify=build['stratum'] if build['stratum'].value_counts().min()>=2 else None)
    except ValueError:
        inner_train, inner_val = train_test_split(build, test_size=0.125, random_state=SEED)
    train_df = df[df.group_id.isin(inner_train.group_id)].reset_index(drop=True)
    val_df = df[df.group_id.isin(inner_val.group_id)].reset_index(drop=True)
    test_df = df[df.group_id.isin(holdout.group_id)].reset_index(drop=True)
    print('NEW group-aware split generated; not identical to the historic split.')

for a,b in [(train_df,val_df),(train_df,test_df),(val_df,test_df)]:
    if 'group_id' in a.columns and 'group_id' in b.columns:
        assert not (set(a.group_id)&set(b.group_id)), 'Group overlap detected'
assert len(train_df) + len(val_df) + len(test_df) == len(metadata)
for name, frame in [('train',train_df),('val',val_df),('test',test_df)]:
    frame.to_csv(SPLIT_DIR / f'{name}.csv',index=False)
print({n:len(f) for n,f in [('train',train_df),('val',val_df),('test',test_df)]})
classes = sorted(metadata.unified_label.unique())
class_to_idx = {c:i for i,c in enumerate(classes)}
display(pd.DataFrame({n:f.unified_label.value_counts().reindex(classes,fill_value=0)
                      for n,f in [('train',train_df),('val',val_df),('test',test_df)]}))

In [ ]:
# 4. Class-distribution figure and example images
counts_plot = metadata.unified_label.value_counts().reindex(classes, fill_value=0)
ax = counts_plot.plot(kind='bar', figsize=(12,4), title='Class distribution — full dataset')
ax.set(xlabel='Class',ylabel='Images'); plt.xticks(rotation=35,ha='right'); plt.tight_layout()
plt.savefig(ARTIFACTS / 'class_distribution.png', dpi=180); plt.show()

fig,axes=plt.subplots(2,4,figsize=(16,8))
for ax,cls in zip(axes.flat,classes):
    row=metadata[metadata.unified_label.eq(cls)].sample(1,random_state=SEED).iloc[0]
    with Image.open(IMAGE_DIR / row.filename) as im:
        ax.imshow(im.convert('RGB'))
    ax.set_title(cls); ax.axis('off')
plt.tight_layout();plt.savefig(ARTIFACTS/'class_examples.png',dpi=150);plt.show()

In [ ]:
# 5. Datasets, transforms and dataloaders
train_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=.15,contrast=.15),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224,224)),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])
class FundusDataset(Dataset):
    def __init__(self, table, image_dir, transform):
        self.table=table.reset_index(drop=True);self.image_dir=Path(image_dir);self.transform=transform
    def __len__(self): return len(self.table)
    def __getitem__(self,i):
        row=self.table.iloc[i]
        with Image.open(self.image_dir/row.filename) as im:
            x=self.transform(im.convert('RGB'))
        return x,class_to_idx[row.unified_label]
train_dataset=FundusDataset(train_df,IMAGE_DIR,train_transform)
val_dataset=FundusDataset(val_df,IMAGE_DIR,eval_transform)
test_dataset=FundusDataset(test_df,IMAGE_DIR,eval_transform)
LOADER_KW = dict(batch_size=32,num_workers=2,pin_memory=torch.cuda.is_available())
train_loader=DataLoader(train_dataset,shuffle=True,**LOADER_KW)
val_loader=DataLoader(val_dataset,shuffle=False,**LOADER_KW)
test_loader=DataLoader(test_dataset,shuffle=False,**LOADER_KW)
print(len(train_dataset),len(val_dataset),len(test_dataset))

In [ ]:
# 6. ResNet18 (ImageNet pretrained), train FC first
try:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
except Exception as e:
    raise RuntimeError('Could not download/load pretrained ResNet18 weights; enable Kaggle Internet or mount the weights.') from e
for p in model.parameters(): p.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,len(classes))
model=model.to(device)
frequencies=train_df.unified_label.value_counts().reindex(classes).to_numpy()
original_weights=len(train_df)/(len(classes)*frequencies)
criterion_stage1=nn.CrossEntropyLoss(weight=torch.tensor(original_weights,dtype=torch.float32,device=device))
opt_stage1=torch.optim.Adam(model.fc.parameters(),lr=1e-3)
print('Trainable:',sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# 7. Reusable training / validation and checkpointing
# Saves best state after EACH validation epoch; optionally resumes on Kaggle reruns
N_CLASSES=len(classes)
def train_validate(model,optimizer,criterion,train_loader,val_loader,epochs,save_name,
                   prior_best=-1,freeze_bn=True):
    history=[];best=prior_best
    save_path=ARTIFACTS/save_name
    if save_path.exists():
        raise FileExistsError(f'{save_path} exists. Rename it or explicitly archive it before rerunning.')
    for epoch in range(1,epochs+1):
        model.train()
        if freeze_bn:
            for layer in model.modules():
                if isinstance(layer,nn.BatchNorm2d):layer.eval()
        sumloss=0;correct=0;n=0
        for x,y in tqdm(train_loader,desc=f'{save_name} epoch {epoch}/{epochs}'):
            x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            out=model(x);loss=criterion(out,y);loss.backward();optimizer.step()
            sumloss+=loss.item()*len(y);correct+=(out.argmax(1)==y).sum().item();n+=len(y)
        model.eval();vloss=0;vy=[];vp=[]
        with torch.inference_mode():
            for x,y in val_loader:
                x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
                out=model(x);vloss+=criterion(out,y).item()*len(y)
                vp.extend(out.argmax(1).cpu().tolist());vy.extend(y.cpu().tolist())
        f1=f1_score(vy,vp,labels=list(range(N_CLASSES)),average='macro',zero_division=0)
        row=dict(epoch=epoch,train_loss=sumloss/n,train_acc=correct/n,
                 val_loss=vloss/len(val_dataset),val_acc=accuracy_score(vy,vp),val_macro_f1=f1)
        history.append(row)
        print({k:round(v,4) for k,v in row.items()})
        if f1>best:
            best=f1
            torch.save(model.state_dict(),save_path)
            print('Saved:',save_path)
        pd.DataFrame(history).to_csv(ARTIFACTS/f'{save_path.stem}_history.csv',index=False)
    return pd.DataFrame(history),best,save_path if save_path.exists() else None

def load_weights(path):
    model.load_state_dict(torch.load(path,map_location=device,weights_only=True))
    model.to(device);model.eval()

In [ ]:
# 8. Stage 1 — FC only, 5 epochs
hist1,best1,stage1_path = train_validate(
    model,opt_stage1,criterion_stage1,train_loader,val_loader,epochs=5,
    save_name='best_resnet18_stage1.pth'
)
assert stage1_path, 'No Stage 1 checkpoint'
print('Best Stage 1 macro F1:',best1)

In [ ]:
# 9. Stage 2 — fine-tune layer4 and FC, 5 epochs
load_weights(stage1_path)
for p in model.parameters():p.requires_grad=False
for p in model.layer4.parameters():p.requires_grad=True
for p in model.fc.parameters():p.requires_grad=True
opt_stage2=torch.optim.Adam([
    {'params':model.layer4.parameters(),'lr':1e-5},
    {'params':model.fc.parameters(),'lr':1e-4},
])
hist2,best2,stage2_path = train_validate(
    model,opt_stage2,criterion_stage1,train_loader,val_loader,epochs=5,
    save_name='best_resnet18_stage2.pth',prior_best=best1
)
print('Stage 2 best:',best2, 'new checkpoint:',stage2_path)

In [ ]:
# 10. Stage 3 — sqrt-inverse-frequency reweighting, 5 epochs
start_stage3 = stage2_path if stage2_path else stage1_path
start_f1 = best2 if stage2_path else best1
load_weights(start_stage3)
for p in model.parameters():p.requires_grad=False
for p in model.layer4.parameters():p.requires_grad=True
for p in model.fc.parameters():p.requires_grad=True
light_weights=np.sqrt(original_weights)
criterion_light=nn.CrossEntropyLoss(
    weight=torch.tensor(light_weights,dtype=torch.float32,device=device))
opt_light=torch.optim.Adam([
    {'params':model.layer4.parameters(),'lr':5e-6},
    {'params':model.fc.parameters(),'lr':5e-5},
])
hist3,best3,stage3_path=train_validate(
    model,opt_light,criterion_light,train_loader,val_loader,epochs=5,
    save_name='best_resnet18_reweighted.pth',prior_best=start_f1
)
SELECTED_PATH = stage3_path if stage3_path else start_stage3
print('Selected checkpoint:',SELECTED_PATH,'Validation Macro F1:',max(start_f1,best3))

## Model selection / independent test

Review validation history and per-class reports before deciding on the final checkpoint and inference method. **Do not tune weights or choose TTA views based on test results.** On a new run, run the held-out test cell once after the choice is final. The historic test results in `results/` are from an earlier session and are not guaranteed to match this reconstructed run.

In [ ]:
# 11. Validation classification report and confusion matrix for chosen checkpoint
load_weights(SELECTED_PATH)
def predict(loader, tta='none'):
    preds=[];targets=[];logits_batches=[]
    black=[-.485/.229,-.456/.224,-.406/.225]
    def rot(x,deg):return TF.rotate(x,deg,interpolation=InterpolationMode.BILINEAR,fill=black)
    with torch.inference_mode():
        for x,y in tqdm(loader,desc=f'Predict ({tta})'):
            x=x.to(device)
            flip=torch.flip(x,dims=[3]) if tta!='none' else None
            if tta=='none': views=[x]
            elif tta=='flip': views=[x,flip]
            elif tta=='four': views=[x,flip,rot(x,5),rot(x,-5)]
            elif tta=='eight':
                views=[x,flip,rot(x,5),rot(x,-5),rot(x,10),rot(x,-10),rot(flip,5),rot(flip,-5)]
            else:raise ValueError(tta)
            scores=torch.stack([model(v) for v in views]).mean(0)
            logits_batches.append(scores.cpu().numpy())
            preds.extend(scores.argmax(1).cpu().tolist());targets.extend(y.tolist())
    return np.array(targets),np.array(preds),np.concatenate(logits_batches)

val_y,val_p,val_logits=predict(val_loader,'none')
print(classification_report(val_y,val_p,labels=list(range(N_CLASSES)),target_names=classes,zero_division=0,digits=4))
cm=confusion_matrix(val_y,val_p,labels=list(range(N_CLASSES)))
fig,ax=plt.subplots(figsize=(11,8));im=ax.imshow(cm,cmap='Blues');fig.colorbar(im,ax=ax)
ax.set(xticks=range(N_CLASSES),yticks=range(N_CLASSES),xticklabels=classes,yticklabels=classes,
       xlabel='Predicted',ylabel='Actual',title='Validation confusion matrix')
plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8)
plt.tight_layout();plt.savefig(ARTIFACTS/'validation_confusion_matrix.png',dpi=160);plt.show()

In [ ]:
# 12. Optional exploratory TTA comparison on VALIDATION ONLY
rows=[]
for method in ['none','flip','four','eight']:
    yy,pp,_=predict(val_loader,method)
    rows.append({'method':method,'accuracy':accuracy_score(yy,pp),
                 'macro_f1':f1_score(yy,pp,labels=list(range(N_CLASSES)),average='macro',zero_division=0)})
tta_validation=pd.DataFrame(rows)
tta_validation.to_csv(ARTIFACTS/'tta_validation_exploratory.csv',index=False)
display(tta_validation)
# NOTE: Later TTA variants are explorations; repeated validation selection can overfit validation.

In [ ]:
# 13. Inspect the 8 transformations — VAL example (no data are modified on disk)
row=val_df.iloc[0]
with Image.open(IMAGE_DIR/row.filename) as im: raw=im.convert('RGB')
flip=TF.hflip(raw)
views=[('Original',raw),('Horizontal flip',flip),
       ('+5°',TF.rotate(raw,5,interpolation=InterpolationMode.BILINEAR,fill=0)),
       ('−5°',TF.rotate(raw,-5,interpolation=InterpolationMode.BILINEAR,fill=0)),
       ('+10°',TF.rotate(raw,10,interpolation=InterpolationMode.BILINEAR,fill=0)),
       ('−10°',TF.rotate(raw,-10,interpolation=InterpolationMode.BILINEAR,fill=0)),
       ('Flip +5°',TF.rotate(flip,5,interpolation=InterpolationMode.BILINEAR,fill=0)),
       ('Flip −5°',TF.rotate(flip,-5,interpolation=InterpolationMode.BILINEAR,fill=0))]
fig,axes=plt.subplots(2,4,figsize=(15,8))
for ax,(name,im) in zip(axes.flat,views):ax.imshow(im);ax.set_title(name);ax.axis('off')
plt.tight_layout();plt.savefig(ARTIFACTS/'8_tta_views_example.png',dpi=150);plt.show()

In [ ]:
# 14. FINAL TEST — run exactly once after freezing model and inference choices
# To use an alternative predefined TTA method, decide it using val BEFORE this cell.
FINAL_TTA='none'  # 'none', 'flip', 'four', or 'eight'
load_weights(SELECTED_PATH)
test_y,test_p,_=predict(test_loader,FINAL_TTA)
summary={'checkpoint':str(SELECTED_PATH),'tta':FINAL_TTA,
         'test_accuracy':accuracy_score(test_y,test_p),
         'test_macro_f1':f1_score(test_y,test_p,labels=list(range(N_CLASSES)),average='macro',zero_division=0)}
print(summary)
report=classification_report(test_y,test_p,labels=list(range(N_CLASSES)),target_names=classes,
                             output_dict=True,zero_division=0)
pd.DataFrame(report).T.to_csv(ARTIFACTS/'test_classification_report.csv')
(ARTIFACTS/'test_summary.json').write_text(json.dumps(summary,indent=2))
cm=confusion_matrix(test_y,test_p,labels=list(range(N_CLASSES)))
fig,ax=plt.subplots(figsize=(11,8));im=ax.imshow(cm,cmap='Blues');fig.colorbar(im,ax=ax)
ax.set(xticks=range(N_CLASSES),yticks=range(N_CLASSES),xticklabels=classes,yticklabels=classes,
       xlabel='Predicted',ylabel='Actual',title='Held-out test confusion matrix')
plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8)
plt.tight_layout();plt.savefig(ARTIFACTS/'test_confusion_matrix.png',dpi=180);plt.show()

In [ ]:
# 15. Inference timing on current GPU (method selected in FINAL_TTA)
# Throughput includes dataloader/transfer overhead. Match this method with your scratch CNN.
load_weights(SELECTED_PATH)
black=[-.485/.229,-.456/.224,-.406/.225]
def forward_tta(x,method):
    if method=='none': return model(x)
    flip=torch.flip(x,dims=[3])
    rot=lambda v,a:TF.rotate(v,a,interpolation=InterpolationMode.BILINEAR,fill=black)
    views={'flip':[x,flip], 'four':[x,flip,rot(x,5),rot(x,-5)],
           'eight':[x,flip,rot(x,5),rot(x,-5),rot(x,10),rot(x,-10),rot(flip,5),rot(flip,-5)]}[method]
    return torch.stack([model(v) for v in views]).mean(0)
with torch.inference_mode():
    xx,_=next(iter(test_loader));xx=xx.to(device)
    for _ in range(5):forward_tta(xx,FINAL_TTA)
    if device.type=='cuda':torch.cuda.synchronize()
    start=time.perf_counter();total=0
    for xx,yy in test_loader:
        xx=xx.to(device);forward_tta(xx,FINAL_TTA);total+=len(yy)
    if device.type=='cuda':torch.cuda.synchronize()
    seconds=time.perf_counter()-start
speed={'tta':FINAL_TTA,'samples':total,'total_seconds':seconds,
       'ms_per_image':seconds*1000/total,'images_per_second':total/seconds}
print(speed)
(ARTIFACTS/'inference_speed.json').write_text(json.dumps(speed,indent=2))

In [ ]:
# 16. Backup everything BEFORE ending the Kaggle session
# Download this ZIP from /kaggle/working; it contains checkpoints, metrics, split CSVs and figures.
import shutil
archive=shutil.make_archive('/kaggle/working/eye_disease_training_outputs','zip',str(ARTIFACTS))
print('DOWNLOAD THIS FILE:',archive)
for p in ARTIFACTS.glob('*.pth'):
    print('CHECKPOINT:',p,p.stat().st_size/1024**2,'MiB')